# CUTEst

In [ ]:
import os

from pathlib import Path
import numpy as np
import pandas as pd

from data.CUTEst.check_CUTEst_problems import problemsToRun
from qnlab.experiment.for_cutest_run import (
    CUTEstTask, get_file_path, load_npz, run_tasks,
)
from qnlab.util.method import Method, get_methods
from qnlab.experiment.for_cutest_vis import draw_pp

## Experiment workflow

This single workflow covers the floating-point experiments and all explicit-noise experiments reported in the paper. Results are stored under `data/temp/<scenario>/seed_<seed>/` and overwrite earlier results for the same condition. A full run is long-running.

In [ ]:
working_directory = Path.cwd().resolve()
if (working_directory / "pyproject.toml").exists():
    repository_root = working_directory
elif (working_directory.parent / "pyproject.toml").exists():
    repository_root = working_directory.parent
else:
    raise RuntimeError("Open this notebook from the repository root or notebooks/.")
os.chdir(repository_root)
print(repository_root)

In [ ]:
SCENARIOS = {
    "float64": {
        "precision": 64, "function_noise": 0.0, "gradient_noise": 0.0,
        "assumed_function_error": None, "solver_gtol": None,
        "score_gtols": [1e-1, 1e-3, 1e-5],
    },
    "float32": {
        "precision": 32, "function_noise": 0.0, "gradient_noise": 0.0,
        "assumed_function_error": None, "solver_gtol": None,
        "score_gtols": [1e-1, 1e-3, 1e-5],
    },
    "float16": {
        "precision": 16, "function_noise": 0.0, "gradient_noise": 0.0,
        "assumed_function_error": None, "solver_gtol": None,
        "score_gtols": [1e-1, 1e-3, 1e-5],
    },
    # Theory-aligned experiment: the gradient oracle is exact.
    "function_only": {
        "precision": 64, "function_noise": 1e-3, "gradient_noise": 0.0,
        "assumed_function_error": 1e-2, "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    # Deliberately adverse stress test outside the exact-gradient theorem.
    "joint_noise": {
        "precision": 64, "function_noise": 1e-3, "gradient_noise": 1e-3,
        "assumed_function_error": 1e-2, "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    "eps_under": {
        "precision": 64, "function_noise": 1e-3, "gradient_noise": 0.0,
        "assumed_function_error": 1e-4, "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    "eps_nominal": {
        "precision": 64, "function_noise": 1e-3, "gradient_noise": 0.0,
        "assumed_function_error": 1e-2, "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    "eps_over": {
        "precision": 64, "function_noise": 1e-3, "gradient_noise": 0.0,
        "assumed_function_error": 1e-1, "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
}

SCENARIOS_TO_RUN = list(SCENARIOS)
SEEDS = [0]
PROBLEMS_TO_RUN = None  # For a quick check, use e.g. ["ROSENBR"].
METHODS_TO_RUN = None  # For a quick check, use e.g. ["NTRQN"].
TIME_LIMIT = 600.0
MAX_ITERATIONS = 15_000
RUN_EXPERIMENTS = False
ERROR_CAUSING_TASKS = [
    (16, "INDEFM", "SciPy"),
    (16, "INDEFM", "NTRQN"),
    (16, "INDEFM", "NTRQN-MS"),
    (16, "INDEFM", "Reg-Sec"),
    (16, "OSCIGRAD", "NTRQN-MS"),
    (32, "INDEFM", "NTRQN"),
    (32, "INDEFM", "NTRQN-MS"),
    (32, "OSCIGRAD", "NTRQN-MS"),
]


## Methods and execution

Leave `RUN_EXPERIMENTS = False` to inspect the selected task list. Set it to `True` only when ready. The two NTQN stopping rules are retained as separate methods until their empirical behavior has been compared.

In [ ]:
STANDARD_LABELS = {
    "NTRQN", "NTRQN-MS", "Line", "Line-MS",
    "Reg", "Reg-Sec", "SciPy", "NTQN",
}
SCENARIO_DEFAULT_LABELS = {
    "function_only": STANDARD_LABELS
    | {"NTRQN-OFFO", "NTRQN-Restart", "NTQN-Default-Termination"},
    "joint_noise": STANDARD_LABELS | {"NTQN-Default-Termination"},
    "eps_under": {"NTRQN", "NTRQN-MS"},
    "eps_nominal": {"NTRQN", "NTRQN-MS"},
    "eps_over": {"NTRQN", "NTRQN-MS"},
}
for scenario in ("float64", "float32", "float16"):
    SCENARIO_DEFAULT_LABELS[scenario] = STANDARD_LABELS

In [ ]:
def get_experiment_methods(max_iterations, solver_gtol=None):
    methods, _, _ = get_methods(m=10, MI=max_iterations)
    methods.extend(
        [
            (
                Method("NTRQN", "cautious", "damped", "bfgs", label="NTRQN-OFFO"),
                {"m": 10, "max_iterations": max_iterations, "force_offo": 1},
            ),
            (
                Method("NTRQN", "cautious", "damped", "bfgs", label="NTRQN-Restart"),
                {"m": 10, "max_iterations": max_iterations, "restart_threshold": 1.0},
            ),
            (
                Method("NTQN", "raw", "raw", "bfgs", label="NTQN-Default-Termination"),
                {"m": 10, "max_iterations": max_iterations, "terminate": 3, "stop_at_gtol": 0},
            ),
        ]
    )
    if solver_gtol is not None:
        methods = [
            (method, option | {"gtol": solver_gtol})
            for method, option in methods
        ]
    return methods


def get_problem_names(precision, selected=None):
    names = problemsToRun(precision)
    if selected is None:
        return names
    unknown = sorted(set(selected) - set(names))
    if unknown:
        raise ValueError(f"Problems not in the {precision}-bit valid list: {unknown}")
    return selected


def make_task(scenario, seed, problem_name, method, options):
    config = SCENARIOS[scenario]
    assumed_error = config["assumed_function_error"]
    return CUTEstTask(
        problem_name=problem_name,
        method=method,
        options=options,
        precision=config["precision"],
        function_noise=np.float64(config["function_noise"]),
        gradient_noise=np.float64(config["gradient_noise"]),
        assumed_function_error=(
            None if assumed_error is None else np.float64(assumed_error)
        ),
        seed=seed,
        scenario=scenario,
    )


def load_scenario_results(scenario, seed, gtol):
    config = SCENARIOS[scenario]
    problems = get_problem_names(config["precision"], PROBLEMS_TO_RUN)
    method_options = get_experiment_methods(MAX_ITERATIONS, config["solver_gtol"])
    labels = METHODS_TO_RUN or SCENARIO_DEFAULT_LABELS[scenario]
    method_options = [entry for entry in method_options if entry[0].label in labels]
    alg_names = [method.label for method, _ in method_options]
    calls = np.full((len(method_options), len(problems)), np.inf)
    for problem_index, problem in enumerate(problems):
        for method_index, (method, options) in enumerate(method_options):
            task = make_task(scenario, seed, problem, method, options)
            callback = load_npz(task, verbose=False)
            reached = np.flatnonzero(np.asarray(callback.gnorms) <= gtol)
            if reached.size > 0:
                calls[method_index, problem_index] = max(
                    1, callback.calls[reached[0]]
                )
    return alg_names, calls, problems

In [ ]:
unknown_scenarios = sorted(set(SCENARIOS_TO_RUN) - set(SCENARIOS))
if unknown_scenarios:
    raise ValueError(f"Unknown scenarios: {unknown_scenarios}")

tasks = []
for scenario in SCENARIOS_TO_RUN:
    config = SCENARIOS[scenario]
    selected_problems = get_problem_names(config["precision"], PROBLEMS_TO_RUN)
    selected_method_options = get_experiment_methods(
        MAX_ITERATIONS, config["solver_gtol"]
    )
    if METHODS_TO_RUN is not None:
        selected_method_options = [
            entry for entry in selected_method_options if entry[0].label in METHODS_TO_RUN
        ]
        missing = sorted(
            set(METHODS_TO_RUN) - {entry[0].label for entry in selected_method_options}
        )
        if missing:
            raise ValueError(f"Unknown method labels: {missing}")
    else:
        selected_method_options = [
            entry
            for entry in selected_method_options
            if entry[0].label in SCENARIO_DEFAULT_LABELS[scenario]
        ]
    tasks.extend(
        make_task(scenario, seed, problem, method, options)
        for seed in SEEDS
        for problem in selected_problems
        for method, options in selected_method_options
    )
print(f"Prepared {len(tasks)} tasks.")
for task in tasks:
    path = get_file_path(task)
    print(
        f"{task.scenario:13s} seed={task.seed} {task.problem_name:24s} "
        f"{task.method.label:18s} -> {path}"
    )

if RUN_EXPERIMENTS:
    run_tasks(
        tasks, ERROR_CAUSING_TASKS, int(TIME_LIMIT), overwrite=True
    )
else:
    print("Dry run only. Set RUN_EXPERIMENTS = True to execute these tasks.")

## Visualize saved results

Select scenarios after their runs have completed. The default empty list avoids overwriting existing figures accidentally.

In [ ]:
SCENARIOS_TO_PLOT = []
PLOT_SEED = 0

_, ALGORITHM_COLORS, ALGORITHM_LINE_STYLES = get_methods()
for scenario in SCENARIOS_TO_PLOT:
    config = SCENARIOS[scenario]
    for gtol in config["score_gtols"]:
        alg_names, calls, problems = load_scenario_results(scenario, PLOT_SEED, gtol)
        output_path = None
        if scenario not in {"float64", "float32", "float16", "joint_noise"}:
            gtol_name = f"{gtol:.0e}".replace("+", "")
            output_path = Path("doc/imgs/compare") / f"_pp_{scenario}_gtol{gtol_name}.pdf"
        draw_pp(
            alg_names,
            calls,
            ALGORITHM_COLORS,
            ALGORITHM_LINE_STYLES,
            config["precision"],
            np.float64(max(config["function_noise"], config["gradient_noise"])),
            np.float64(gtol),
            output_path=output_path,
        )

        table = pd.DataFrame(calls.T, index=problems, columns=alg_names)
        display(table)